<a href="https://colab.research.google.com/github/CopingMoa/AI_ASSISTED_APP_TEMPLATE/blob/main/ensemble_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import glob
import os
import pandas as pd

def combine_category_folders(root_path, output_path, log_path="combine_log.json"):
    """
    Expects structure:
        root_path/
            Benign/part-*.csv
            Reconnaissance/part-*.csv
            Exploitation/part-*.csv
            ...
    """
    import json

    category_folders = sorted(
        [d for d in glob.glob(os.path.join(root_path, "*")) if os.path.isdir(d)]
    )

    if not category_folders:
        raise FileNotFoundError(f"No category subfolders found in {root_path}")

    dfs = []
    log = []
    reference_columns = None

    for folder in category_folders:
        category_name = os.path.basename(folder)  # "Benign", "Reconnaissance", etc.
        part_files = sorted(glob.glob(os.path.join(folder, "part-*.csv")))

        if not part_files:
            print(f"[!] No part-*.csv files in {folder}, skipping")
            continue

        category_row_count = 0

        for f in part_files:
            df = pd.read_csv(f)

            # Schema check — must match across ALL categories, not just within one
            if reference_columns is None:
                reference_columns = list(df.columns)
            elif list(df.columns) != reference_columns:
                raise ValueError(
                    f"Schema mismatch in {f}\n"
                    f"Expected: {reference_columns}\n"
                    f"Got: {list(df.columns)}"
                )

            # This is the critical line — category comes from the FOLDER,
            # not from the CSV content, since these files may have no
            # label column of their own (as you found earlier with the
            # unlabeled Benign partition).
            df["source_category"] = category_name

            dfs.append(df)
            category_row_count += len(df)

        log.append({"category": category_name, "files": len(part_files), "rows": category_row_count})
        print(f"[+] {category_name}: {len(part_files)} files, {category_row_count:,} rows")

    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv(output_path, index=False)

    with open(log_path, "w") as out:
        json.dump({"categories": log, "total_rows": len(combined)}, out, indent=2)

    print(f"[+] Combined {len(category_folders)} categories -> {len(combined):,} total rows -> {output_path}")
    return combined

combined_df = combine_category_folders("raw_parts/", "uwf_zeekdata24_combined.csv")

FileNotFoundError: No category subfolders found in raw_parts/